In [ ]:
# Plotting all catchments in this folder 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import numpy as np
import pandas as pd
import xarray as xr
import itertools
import geopandas as gpd
import re

import pickle

# from matplotlib_venn import venn3
import matplotlib.pyplot as plt

In [ ]:
import torch
import gc

# File paths
files = {
    'train_res': '/perm/mokr/CamelsGB_V2/train_res_tensor_data.pt',
    'train_non_res': '/perm/mokr/CamelsGB_V2/train_non_res_tensor_data.pt',
    'val_res': '/perm/mokr/CamelsGB_V2/val_res_tensor_data.pt',
    'val_non_res': '/perm/mokr/CamelsGB_V2/val_non_res_tensor_data.pt',
    'test_res': '/perm/mokr/CamelsGB_V2/test_res_tensor_data.pt',
    'test_non_res': '/perm/mokr/CamelsGB_V2/test_non_res_tensor_data.pt'
}

# Load and print lengths
train_res_tensor_data = torch.load(files['train_res'])
train_non_res_tensor_data = torch.load(files['train_non_res'])
print(f"Train: {len(train_res_tensor_data)} reservoir, {len(train_non_res_tensor_data)} non-reservoir")
del train_res_tensor_data, train_non_res_tensor_data

val_res_tensor_data = torch.load(files['val_res'])
val_non_res_tensor_data = torch.load(files['val_non_res'])
print(f"Val: {len(val_res_tensor_data)} reservoir, {len(val_non_res_tensor_data)} non-reservoir")
del val_res_tensor_data, val_non_res_tensor_data


test_res_tensor_data = torch.load(files['test_res'])
test_non_res_tensor_data = torch.load(files['test_non_res'])

print(f"Test: {len(test_res_tensor_data)} reservoir, {len(test_non_res_tensor_data)} non-reservoir")

# Delete objects from memory
# del test_res_tensor_data, test_non_res_tensor_data

# # Run garbage collection to free memory
# gc.collect()
# print("Objects deleted and garbage collection completed")

# CamelsGB Reservoirs

In [ ]:
# Creating a single CSV file of variables

folder_path = '/perm/mokr/CamelsGB_V2/Catchment_Attributes/'
All_Static_Variables = pd.DataFrame()
# List CSV files directly in the main folder (excluding subfolders)
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# Loop through the CSV files
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    # Read the CSV file and concatenate it with the merged_df
    static_variable_group = pd.read_csv(file_path)
    All_Static_Variables = pd.concat([All_Static_Variables, static_variable_group], ignore_index=True)

# Merge by 'gauge_id'
All_Static_Variables = All_Static_Variables.groupby('gauge_id').sum().reset_index()

In [ ]:
# Camels2_Human = pd.read_csv('/perm/mokr/CamelsGB_V2/Catchment_Attributes/camels_gb_v2_humaninfluence_attributes.csv')


In [ ]:
All_Static_Variables['reservoir_contributing_area'].value_counts().sort_index()
print(All_Static_Variables['num_reservoir'].value_counts().sort_index())

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Load the UK country boundary shapefile
uk_boundary = gpd.read_file('/home/mokr/Camels_GB/GB_Shape_Files/CTRY_DEC_2024_UK_BFC.shp')

# Create gauge GeoDataFrame
gdf_gauges = gpd.GeoDataFrame(
    All_Static_Variables,
    geometry=gpd.points_from_xy(
        All_Static_Variables['gauge_lon'], 
        All_Static_Variables['gauge_lat']
    ),
    crs='EPSG:4326'
)

# Reproject gauges to match shapefile CRS
if uk_boundary.crs != gdf_gauges.crs:
    gdf_gauges = gdf_gauges.to_crs(uk_boundary.crs)

# Create discrete categories
def categorize_reservoirs(num):
    if num == 0:
        return '0'
    elif num == 1:
        return '1'
    elif 2 <= num <= 4:
        return '2-4'
    elif 5 <= num <= 9:
        return '5-9'
    else:
        return '>=10'

gdf_gauges['reservoir_category'] = gdf_gauges['num_reservoir'].apply(categorize_reservoirs)

# Define colors for each category
category_colors = {
    '0': '#440154',      # Dark purple
    '1': '#3b528b',      # Blue
    '2-4': '#21918c',    # Teal
    '5-9': '#5ec962',   # Green
    '>=10': '#fde725'     # Yellow
}

# Map colors to the dataframe
gdf_gauges['color'] = gdf_gauges['reservoir_category'].map(category_colors)

# Create the map
fig, ax = plt.subplots(figsize=(12, 14))

# Plot UK boundary
uk_boundary.plot(ax=ax, color='lightgray', edgecolor='darkgray', linewidth=1.5)

# Plot gauges with discrete colors
gdf_gauges.plot(
    ax=ax,
    color=gdf_gauges['color'],
    markersize=35,
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5
)

# Create custom legend
legend_elements = [
    Patch(facecolor=category_colors['0'], edgecolor='black', label='0 reservoirs'),
    Patch(facecolor=category_colors['1'], edgecolor='black', label='1 reservoir'),
    Patch(facecolor=category_colors['2-4'], edgecolor='black', label='2-4 reservoirs'),
    Patch(facecolor=category_colors['5-9'], edgecolor='black', label='5-9 reservoirs'),
    Patch(facecolor=category_colors['>=10'], edgecolor='black', label='>=10 reservoirs')
]
ax.legend(handles=legend_elements, loc='upper left', title='Number of Reservoirs', 
          frameon=True, fancybox=True, shadow=True)

# Customize the plot
ax.set_title('Gauge Locations by Number of Reservoirs', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Easting', fontsize=12)
ax.set_ylabel('Northing', fontsize=12)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nValue counts by category:")
print(gdf_gauges['reservoir_category'].value_counts().sort_index())

# Optional: Print what countries are in the shapefile
print(f"\nCountries in shapefile:")
if 'CTRY24NM' in uk_boundary.columns:
    print(uk_boundary['CTRY24NM'].tolist())
elif 'NAME' in uk_boundary.columns:
    print(uk_boundary['NAME'].tolist())

In [ ]:
# Create gauge GeoDataFrame using Camels2_Human
gdf_gauges = gpd.GeoDataFrame(
    All_Static_Variables,
    geometry=gpd.points_from_xy(
        All_Static_Variables['gauge_lon'], 
        All_Static_Variables['gauge_lat']
    ),
    crs='EPSG:4326'
)

# Reproject gauges to match shapefile CRS
if uk_boundary.crs != gdf_gauges.crs:
    gdf_gauges = gdf_gauges.to_crs(uk_boundary.crs)

# Create discrete categories for reservoir_contributing_area
def categorize_reservoir_area(value):
    if value == 0:
        return '0'
    elif 0 < value <= 10:
        return '0-10'
    elif 10 < value <= 25:
        return '10-25'
    elif 25 < value <= 75:
        return '25-75'
    else:
        return '75-100'

gdf_gauges['area_category'] = gdf_gauges['reservoir_contributing_area'].apply(categorize_reservoir_area)

# Define colors for each category
category_colors = {
    '0': '#440154',      # Dark purple
    '0-10': '#3b528b',   # Blue
    '10-25': '#21918c',  # Teal
    '25-75': '#5ec962',  # Green
    '75-100': '#fde725'  # Yellow
}

# Map colors to the dataframe
gdf_gauges['color'] = gdf_gauges['area_category'].map(category_colors)

# Create the map
fig, ax = plt.subplots(figsize=(12, 14))

# Plot UK boundary
uk_boundary.plot(ax=ax, color='lightgray', edgecolor='darkgray', linewidth=1.5)

# Plot gauges with discrete colors
gdf_gauges.plot(
    ax=ax,
    color=gdf_gauges['color'],
    markersize=35,
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5
)

# Create custom legend
legend_elements = [
    Patch(facecolor=category_colors['0'], edgecolor='black', label='0%'),
    Patch(facecolor=category_colors['0-10'], edgecolor='black', label='0-10%'),
    Patch(facecolor=category_colors['10-25'], edgecolor='black', label='10-25%'),
    Patch(facecolor=category_colors['25-75'], edgecolor='black', label='25-75%'),
    Patch(facecolor=category_colors['75-100'], edgecolor='black', label='75-100%')
]
ax.legend(handles=legend_elements, loc='upper left', title='Reservoir Contributing Area (%)', 
          frameon=True, fancybox=True, shadow=True)

# Customize the plot
ax.set_title('Gauge Locations by Reservoir Contributing Area', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Easting', fontsize=12)
ax.set_ylabel('Northing', fontsize=12)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()


print(f"\nValue counts by category:")
print(gdf_gauges['area_category'].value_counts().sort_index())

In [ ]:
# Filter rows where reservoir_contributing_area > 0
reservoir_gauges = All_Static_Variables[
    All_Static_Variables['reservoir_contributing_area'] > 0
].copy()

# Select the specified columns
columns_to_keep = [
    'gauge_id',
    'gauge_lat',
    'gauge_lon',
    'reservoir_contributing_area',
    'reservoir_year_last',
    'daily_flow_period_start',
    'daily_flow_period_end',
    'daily_flow_perc_complete',
    'daily_flow_period_end' 
]

# Create the new table
reservoir_table = reservoir_gauges[columns_to_keep]

# Sort by reservoir_contributing_area (descending) to see highest influence first
reservoir_table = reservoir_table.sort_values('reservoir_contributing_area', ascending=False)

# Reset index for cleaner display
reservoir_table = reservoir_table.reset_index(drop=True)

# Display information about the table
print(f"Total gauges with reservoir influence: {len(reservoir_table)}")
print(f"\nSummary statistics:")
print(reservoir_table.describe())

print(f"\nFirst 10 rows:")
print(reservoir_table.head(10))

# Show distribution by reservoir_contributing_area ranges
print(f"\nDistribution by reservoir contribution:")
print(f"0-10%: {len(reservoir_table[reservoir_table['reservoir_contributing_area'] <= 10])}")
print(f"10-25%: {len(reservoir_table[(reservoir_table['reservoir_contributing_area'] > 10) & (reservoir_table['reservoir_contributing_area'] <= 25)])}")
print(f"25-75%: {len(reservoir_table[(reservoir_table['reservoir_contributing_area'] > 25) & (reservoir_table['reservoir_contributing_area'] <= 75)])}")
print(f"75-100%: {len(reservoir_table[reservoir_table['reservoir_contributing_area'] > 75])}")

In [ ]:
reservoir_table

latest_reservoir_idx = reservoir_table['reservoir_year_last'].idxmax()
latest_reservoir_row = reservoir_table.loc[latest_reservoir_idx]
latest_reservoir_row

earliest_flow_end_idx = reservoir_table['daily_flow_period_end'].idxmin()
earliest_flow_end_row = reservoir_table.loc[earliest_flow_end_idx]
earliest_flow_end_row

In [ ]:
All_Static_Variables
numeric_cols = All_Static_Variables.select_dtypes(include=[np.number]).columns.tolist()


# Normalising Static Data

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Make a copy to avoid modifying original
All_Static_Variables_normalized = All_Static_Variables.copy()

# Define columns to exclude from normalization
exclude_cols = ['gauge_id', 'gauge_name', 'gauge_lat', 'gauge_lon']

# Get columns to normalize
numeric_cols = All_Static_Variables.select_dtypes(include=[np.number]).columns.tolist()
cols_to_normalize = [col for col in numeric_cols if col not in exclude_cols]

print(f"Total columns: {len(All_Static_Variables.columns)}")
print(f"Columns to normalize: {len(cols_to_normalize)}")
print(f"Excluded columns: {exclude_cols}")

# OPTION 1: StandardScaler (Z-score normalization: mean=0, std=1)
# Best for most ML models, handles outliers reasonably
scaler = StandardScaler()
All_Static_Variables_normalized[cols_to_normalize] = scaler.fit_transform(
    All_Static_Variables[cols_to_normalize]
)

print("\n" + "="*60)
print("Normalization complete using StandardScaler")
print("="*60)

# Show before/after statistics for a few columns
sample_cols = cols_to_normalize[:3]  # First 3 normalized columns
print(f"\nBefore normalization (first 3 numeric columns):")
print(All_Static_Variables[sample_cols].describe())

print(f"\nAfter normalization (first 3 numeric columns):")
print(All_Static_Variables_normalized[sample_cols].describe())

# Check for any NaN values introduced
nan_counts = All_Static_Variables_normalized[cols_to_normalize].isna().sum()
if nan_counts.sum() > 0:
    print(f"\nWarning: NaN values found in normalized data:")
    print(nan_counts[nan_counts > 0])
else:
    print(f"\nNo NaN values introduced during normalization ✓")

# Optional: Save the normalized dataframe
# All_Static_Variables_normalized.to_csv('/perm/mokr/Scaled_CamelsGB_V2/normalised_static_data.csv', index=False)

# joblib.dump(scaler, '/perm/mokr/Scaled_CamelsGB_V2/static_vars_scaler.pkl')
# print("\nScaler saved to: /perm/mokr/Scaled_CamelsGB_V2static_vars_scaler.pkl")
# print("(Use this to normalize new data the same way)")


In [ ]:
All_Static_Variables_normalized[numeric_cols].mean()

# Pairing Reservoir and Non Reservoir Catchments

In [ ]:
from scipy.spatial.distance import cdist

# Separate reservoir and non-reservoir catchments
reservoir_gauges = gdf_gauges[gdf_gauges['reservoir_contributing_area'] > 0].copy()
reservoir_gauges = reservoir_gauges.sort_values('gauge_lat').reset_index(drop=True)
non_reservoir_gauges = gdf_gauges[gdf_gauges['reservoir_contributing_area'] == 0].copy()

print(f"Reservoir catchments: {len(reservoir_gauges)}")
print(f"Non-reservoir catchments: {len(non_reservoir_gauges)}")

# Extract coordinates for distance calculation
reservoir_coords = np.column_stack([
    reservoir_gauges.geometry.x, 
    reservoir_gauges.geometry.y
])
non_reservoir_coords = np.column_stack([
    non_reservoir_gauges.geometry.x, 
    non_reservoir_gauges.geometry.y
])

# Create pairing lists
paired_reservoir_indices = []
paired_non_reservoir_indices = []
available_non_reservoir = set(range(len(non_reservoir_gauges)))

# For each reservoir gauge, find closest available non-reservoir gauge
for i in range(len(reservoir_gauges)):
    if not available_non_reservoir:
        print(f"Warning: Ran out of non-reservoir gauges at iteration {i}")
        break
    
    # Calculate distances to all available non-reservoir gauges
    res_coord = reservoir_coords[i:i+1]
    available_indices = list(available_non_reservoir)
    available_coords = non_reservoir_coords[available_indices]
    
    distances = cdist(res_coord, available_coords, metric='euclidean')[0]
    
    # Find closest available gauge
    closest_idx_in_available = np.argmin(distances)
    closest_idx = available_indices[closest_idx_in_available]
    
    # Record pairing
    paired_reservoir_indices.append(i)
    paired_non_reservoir_indices.append(closest_idx)
    
    # Remove from available set
    available_non_reservoir.remove(closest_idx)

print(f"\nSuccessfully paired {len(paired_reservoir_indices)} catchments")

# Create paired dataframes
paired_reservoir = reservoir_gauges.iloc[paired_reservoir_indices].copy()
paired_non_reservoir = non_reservoir_gauges.iloc[paired_non_reservoir_indices].copy()
unpaired_non_reservoir = non_reservoir_gauges.iloc[list(available_non_reservoir)].copy()

# Add pairing information
paired_reservoir['pair_id'] = range(len(paired_reservoir_indices))
paired_non_reservoir['pair_id'] = range(len(paired_non_reservoir_indices))

In [ ]:
# Create map
fig, ax = plt.subplots(figsize=(12, 14))

# Plot UK boundary
uk_boundary.plot(ax=ax, color='lightgray', edgecolor='darkgray', linewidth=1.5)

# Plot unpaired (grey)
unpaired_non_reservoir.plot(
    ax=ax,
    color='white',
    markersize=25,
    alpha=0.4,
    edgecolor='grey',
    linewidth=0.3,
    label='Unpaired non-reservoir'
)

# Plot paired non-reservoir (blue)
paired_non_reservoir.plot(
    ax=ax,
    color='dodgerblue',
    markersize=35,
    alpha=0.5,
    edgecolor='navy',
    linewidth=0.6,
    label='Paired non-reservoir'
)

# Plot paired reservoir (red)
paired_reservoir.plot(
    ax=ax,
    color='red',
    markersize=35,
    alpha=0.5,
    edgecolor='darkred',
    linewidth=0.6,
    label='Reservoir catchment'
)

# Draw lines connecting pairs
for res_idx, non_res_idx in zip(paired_reservoir_indices, paired_non_reservoir_indices):
    res_point = reservoir_coords[res_idx]
    non_res_point = non_reservoir_coords[non_res_idx]
    ax.plot(
        [res_point[0], non_res_point[0]], 
        [res_point[1], non_res_point[1]], 
        color='black', 
        alpha=0.6, 
        linewidth=0.5,
        zorder=1
    )

# Customize plot
ax.legend(loc='upper left', frameon=True, fancybox=True, shadow=True, fontsize=14)
ax.set_title('Paired Reservoir and Non-Reservoir Catchments', 
             fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel('Easting', fontsize=16)
ax.set_ylabel('Northing', fontsize=16)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Print pairing statistics
print(f"\nPairing Statistics:")
print(f"Paired reservoir catchments: {len(paired_reservoir)}")
print(f"Paired non-reservoir catchments: {len(paired_non_reservoir)}")
print(f"Unpaired non-reservoir catchments: {len(unpaired_non_reservoir)}")

# Calculate and display pairing distances
distances = []
for res_idx, non_res_idx in zip(paired_reservoir_indices, paired_non_reservoir_indices):
    dist = np.linalg.norm(reservoir_coords[res_idx] - non_reservoir_coords[non_res_idx])
    distances.append(dist)

distances = np.array(distances)
print(f"\nPairing distance statistics (in CRS units):")
print(f"Mean distance: {distances.mean():.2f}")
print(f"Median distance: {np.median(distances):.2f}")
print(f"Min distance: {distances.min():.2f}")
print(f"Max distance: {distances.max():.2f}")

# Create output dataframe with pairs
pairs_df = paired_reservoir.reset_index()[['gauge_id', 'gauge_lat', 'gauge_lon', 'reservoir_contributing_area', 'gauge_name']].copy()
pairs_df.columns = ['reservoir_gauge_id', 'reservoir_lat', 'reservoir_lon', 'reservoir_contributing_area', 'reservoir_gauge_name']
pairs_df['non_reservoir_gauge_id'] = paired_non_reservoir.reset_index()['gauge_id'].values
pairs_df['non_reservoir_lat'] = paired_non_reservoir.reset_index()['gauge_lat'].values
pairs_df['non_reservoir_lon'] = paired_non_reservoir.reset_index()['gauge_lon'].values
pairs_df['non_reservoir_distance'] = distances
pairs_df['non_reservoir_gauge_name'] = paired_non_reservoir.reset_index()['gauge_name'].values


print("\nFirst 5 pairs:")
print(pairs_df.head())

# Choosing Training, Validation, Test

In [ ]:
pairs_df.columns

In [ ]:
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

def create_geographic_split(pairs_df, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2, random_state=42):
    """
    Create geographically separated train/val/test splits
    """
    n_pairs = len(pairs_df)
    target_train = int(train_ratio * n_pairs)
    target_val = int(val_ratio * n_pairs)
    target_test = n_pairs - target_train - target_val
    
    coords = pairs_df[['reservoir_lat', 'reservoir_lon']].values
    
    # Try different numbers of clusters and select best
    best_split = None
    best_score = float('inf')
    
    for n_clusters in range(3, 8):
        kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=50)
        labels = kmeans.fit_predict(coords)
        
        # Group clusters to match target sizes
        cluster_sizes = pd.Series(labels).value_counts().to_dict()
        
        # Try to assign clusters to splits
        from itertools import combinations
        
        all_clusters = list(cluster_sizes.keys())
        
        # Try all possible ways to partition clusters into 3 groups
        best_partition = None
        min_diff = float('inf')
        
        for train_clusters in combinations(all_clusters, len(all_clusters)//2 + 1):
            train_size = sum(cluster_sizes[c] for c in train_clusters)
            remaining = [c for c in all_clusters if c not in train_clusters]
            
            for val_clusters in combinations(remaining, len(remaining)//2):
                val_size = sum(cluster_sizes[c] for c in val_clusters)
                test_clusters = [c for c in remaining if c not in val_clusters]
                test_size = sum(cluster_sizes[c] for c in test_clusters)
                
                # Calculate how close to target
                diff = (abs(train_size - target_train) + 
                       abs(val_size - target_val) + 
                       abs(test_size - target_test))
                
                if diff < min_diff:
                    min_diff = diff
                    best_partition = (train_clusters, val_clusters, test_clusters)
        
        if min_diff < best_score:
            best_score = min_diff
            best_split = (labels, best_partition)
    
    # Apply best split
    labels, (train_clusters, val_clusters, test_clusters) = best_split
    
    pairs_df['split'] = 'train'
    pairs_df.loc[[i for i, l in enumerate(labels) if l in val_clusters], 'split'] = 'val'
    pairs_df.loc[[i for i, l in enumerate(labels) if l in test_clusters], 'split'] = 'test'
    
    return pairs_df

pairs_df = create_geographic_split(pairs_df)

In [ ]:

from shapely.geometry import Point

# Create GeoDataFrame from pairs_df
geometry = [Point(lon, lat) for lon, lat in zip(pairs_df['reservoir_lon'], pairs_df['reservoir_lat'])]
pairs_gdf = gpd.GeoDataFrame(pairs_df, geometry=geometry, crs='EPSG:4326')

# Convert to same CRS as uk_boundary (likely British National Grid)
pairs_gdf = pairs_gdf.to_crs(uk_boundary.crs)

# Now plot
fig, ax = plt.subplots(figsize=(10, 12))

# Plot UK boundary first
uk_boundary.plot(ax=ax, color='lightgray', edgecolor='darkgray', linewidth=1.5)

# Plot each split with different colors using GeoDataFrame
colors = {'train': 'blue', 'val': 'green', 'test': 'red'}
for split in ['train', 'val', 'test']:
    subset = pairs_gdf[pairs_gdf['split'] == split]
    subset.plot(ax=ax, color=colors[split], 
                label=f'{split} ({len(subset)} pairs)', 
                alpha=0.6, markersize=50)

ax.set_xlabel('Easting', fontsize=14)
ax.set_ylabel('Northing', fontsize=14)
ax.set_title('Geographic Distribution of Train/Val/Test Splits', fontsize=16, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
# plt.savefig('split_visualization.png', dpi=300)
plt.show()

# Print split statistics
for split in ['train', 'val', 'test']:
    subset = pairs_gdf[pairs_gdf['split'] == split]
    print(f"\n{split.upper()}: {len(subset)} pairs ({len(subset)/len(pairs_gdf)*100:.1f}%)")

In [ ]:
with open('/home/mokr/Reservoir_Modelling/Sample_Pairs/catchment_date_pairs.pkl', 'rb') as f:
    catchment_pairings = pickle.load(f)

In [ ]:
def create_catchment_splits(catchment_pairings, pairs_gdf):
    """
    Create split dictionaries matching the catchment_pairings structure
    """
    # Create mapping from gauge_id to split
    pairs_gdf['reservoir_gauge_id'] = pairs_gdf['reservoir_gauge_id'].astype(int).astype(str)
    pairs_gdf['non_reservoir_gauge_id'] = pairs_gdf['non_reservoir_gauge_id'].astype(int).astype(str)
  
    reservoir_split_map = dict(zip(pairs_gdf['reservoir_gauge_id'], pairs_gdf['split']))
    non_reservoir_split_map = dict(zip(pairs_gdf['non_reservoir_gauge_id'], pairs_gdf['split']))

    
    catchment_splits = {
        'reservoir_train': [],
        'reservoir_val': [],
        'reservoir_test': [],
        'non_reservoir_train': [],
        'non_reservoir_val': [],
        'non_reservoir_test': []
    }
    
    # Process paired reservoir entries
    for entry in catchment_pairings['paired_reservoir']:
        gauge_id = str(entry['gauge_id'])
        
        split = reservoir_split_map.get(gauge_id)
        
        if split:
            catchment_splits[f'reservoir_{split}'].append(entry)
    
    # Process paired non-reservoir entries
    for entry in catchment_pairings['paired_non_reservoir']:
        gauge_id = str(entry['gauge_id'])
        split = non_reservoir_split_map.get(gauge_id)
        if split:
            catchment_splits[f'non_reservoir_{split}'].append(entry)
    
    # Print statistics
    for key in catchment_splits.keys():
        print(f"{key}: {len(catchment_splits[key])} entries")
    
    return catchment_splits

# Create the splits
catchment_splits = create_catchment_splits(catchment_pairings, pairs_gdf)

# Save to file
output_path = '/home/mokr/Reservoir_Modelling/Sample_Pairs/catchment_date_pairs_splits.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(catchment_splits, f)
print(f"\nSaved to {output_path}")

# Verify the structure
print(f"\nExample entry from reservoir_train:")
print(catchment_splits['reservoir_train'][0])

# Get Data Range

In [ ]:
with open('/home/mokr/Reservoir_Modelling/Normalised_Temporal_Data.pkl', 'rb') as f:
    temporal_data = pickle.load(f)


paired_reservoir 
paired_non_reservoir
unpaired_non_reservoir

In [ ]:
def get_actual_data_range(gauge_id, temporal_data):
    """Get the actual start and end dates from the timeseries data"""
    if gauge_id not in temporal_data:
        print("Gauge not in dataset")
        return None, None
    
    df = temporal_data[gauge_id]
    if df is None or len(df) == 0:
        print("Gauge data empty")
        return None, None

    dates = pd.to_datetime(df['date'])
    return dates.min(), dates.max()

def check_for_nans_in_range(df, start_date, end_date):
    """Check if there are any NaNs in the specified date range, and adjust bounds if needed"""
    if df is None or len(df) == 0:
        return None, None

    dates = pd.to_datetime(df['date'])
    df_range = df[(dates >= start_date) & (dates <= end_date)]
    
    if len(df_range) == 0:
        return None, None
    
    # Check if there are any NaNs
    if df_range.isna().any().any():
        print(f'There are nans for this catchment')
        # There are NaNs, so filter to complete rows only
        complete_rows = df_range.dropna()
        if len(complete_rows) == 0:
            return None, None
        
        dates = pd.to_datetime(complete_rows['date'])
        return dates.min(), dates.max()
    else:
        # No NaNs in this range, return original bounds
        return start_date, end_date

def find_longest_consecutive_period(df, start_date, end_date):
    """Find the longest consecutive period without NaNs in the specified date range"""
    if df is None or len(df) == 0:
        return None, None
    
    dates = pd.to_datetime(df['date'])
    df_range = df[(dates >= start_date) & (dates <= end_date)].copy()
    df_range['date_col'] = dates[(dates >= start_date) & (dates <= end_date)]
    
    if len(df_range) == 0:
        return None, None
    
    # Check if there are any NaNs
    if not df_range.isna().any().any():
        # No NaNs in this range, return original bounds
        return start_date, end_date
    
    # Find rows with complete data (no NaNs)
    has_complete_data = ~df_range.drop(columns=['date_col']).isna().any(axis=1)
    
    if not has_complete_data.any():
        return None, None
    
    # Find consecutive groups of complete data
    # Create group IDs where each consecutive run gets the same ID
    df_range['complete'] = has_complete_data
    df_range['group'] = (df_range['complete'] != df_range['complete'].shift()).cumsum()
    
    # Filter to only complete data groups
    complete_groups = df_range[df_range['complete']].groupby('group')
    
    # Find the longest group
    longest_group = None
    longest_duration = pd.Timedelta(0)
    
    for group_id, group_df in complete_groups:
        group_start = group_df['date_col'].min()
        group_end = group_df['date_col'].max()
        duration = group_end - group_start
        
        if duration > longest_duration:
            longest_duration = duration
            longest_group = (group_start, group_end)
    
    if longest_group is None:
        return None, None
    
    return longest_group[0], longest_group[1]


def determine_valid_period(gauge_row, temporal_data, is_reservoir=False):
    """Determine valid start and end dates for a gauge"""
    gauge_id = int(gauge_row['gauge_id'])
    
    # Get actual data range
    valid_start, valid_end = get_actual_data_range(f'{gauge_id}', temporal_data)
    
    if valid_start is None or valid_end is None:
        return None, None
    
    # Start with flow period bounds
    flow_start = pd.to_datetime(gauge_row['daily_flow_period_start'])
    flow_end = pd.to_datetime(gauge_row['daily_flow_period_end'])
    
    # For reservoirs, also consider reservoir_year_last
    if is_reservoir and pd.notna(gauge_row.get('reservoir_year_last')):
        reservoir_year = int(gauge_row['reservoir_year_last'])
        # Start from January 1st of the year after last reservoir
        reservoir_date = pd.Timestamp(year=reservoir_year + 1, month=1, day=1)
        valid_start = max(valid_start, reservoir_date)
    
    # Valid end is the earliest of: actual data end, flow period end
    valid_end = flow_end
    
    # Check if valid range exists
    if valid_start >= valid_end:
        return None, None
    
    # final_start, final_end = check_for_nans_in_range(temporal_data[f'{gauge_id}'], valid_start, valid_end)
    final_start, final_end = find_longest_consecutive_period(temporal_data[f'{gauge_id}'], valid_start, valid_end)
    
    return final_start, final_end


In [ ]:
# Process paired reservoir gauges
print("\n" + "="*60)
print("Processing paired reservoir gauges...")
print("="*60)

paired_reservoir['valid_start'] = None
paired_reservoir['valid_end'] = None

for idx, row in paired_reservoir.iterrows():
    start, end = determine_valid_period(row, temporal_data, is_reservoir=True)
    paired_reservoir.at[idx, 'valid_start'] = start
    paired_reservoir.at[idx, 'valid_end'] = end


# Process paired non-reservoir gauges
print("\nProcessing paired non-reservoir gauges...")

paired_non_reservoir['valid_start'] = None
paired_non_reservoir['valid_end'] = None

for idx, row in paired_non_reservoir.iterrows():
    start, end = determine_valid_period(row, temporal_data, is_reservoir=False)
    paired_non_reservoir.at[idx, 'valid_start'] = start
    paired_non_reservoir.at[idx, 'valid_end'] = end

# Align dates for paired gauges
print("\nAligning dates for paired gauges...")

paired_reservoir['pair_start'] = None
paired_reservoir['pair_end'] = None
paired_non_reservoir['pair_start'] = None
paired_non_reservoir['pair_end'] = None

for pair_id in range(len(paired_reservoir)):
    res_row = paired_reservoir[paired_reservoir['pair_id'] == pair_id].iloc[0]
    non_res_row = paired_non_reservoir[paired_non_reservoir['pair_id'] == pair_id].iloc[0]
    
    # Get valid periods for both
    res_start = res_row['valid_start']
    res_end = res_row['valid_end']
    non_res_start = non_res_row['valid_start']
    non_res_end = non_res_row['valid_end']
    
    # Find overlap (latest start, earliest end)
    pair_start = max(res_start, non_res_start)
    pair_end = min(res_end, non_res_end)
    
    # Check if valid overlap exists
    if pair_start < pair_end:
        paired_reservoir.loc[paired_reservoir['pair_id'] == pair_id, 'pair_start'] = pair_start
        paired_reservoir.loc[paired_reservoir['pair_id'] == pair_id, 'pair_end'] = pair_end
        paired_non_reservoir.loc[paired_non_reservoir['pair_id'] == pair_id, 'pair_start'] = pair_start
        paired_non_reservoir.loc[paired_non_reservoir['pair_id'] == pair_id, 'pair_end'] = pair_end
    else:
        print(f"Warning: No valid overlap for pair {pair_id}")

# Process unpaired non-reservoir gauges
print("\nProcessing unpaired non-reservoir gauges...")

unpaired_non_reservoir['valid_start'] = None
unpaired_non_reservoir['valid_end'] = None

for idx, row in unpaired_non_reservoir.iterrows():
    start, end = determine_valid_period(row, temporal_data, is_reservoir=False)
    unpaired_non_reservoir.at[idx, 'valid_start'] = start
    unpaired_non_reservoir.at[idx, 'valid_end'] = end

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

# Paired gauges
valid_pairs = paired_reservoir[paired_reservoir['pair_start'].notna()]
print(f"\nPaired gauges with valid overlapping data: {len(valid_pairs)} out of {len(paired_reservoir)}")

if len(valid_pairs) > 0:
    paired_reservoir['pair_duration_days'] = (
        paired_reservoir['pair_end'] - paired_reservoir['pair_start']
    ).apply(lambda x: x.days if pd.notna(x) else None)
    paired_reservoir['pair_duration_years'] = paired_reservoir['pair_duration_days'] / 365.25
    
    # Filter out None/NaN values for statistics
    valid_durations = paired_reservoir['pair_duration_years'].dropna()
    valid_starts = paired_reservoir['pair_start'].dropna()
    valid_ends = paired_reservoir['pair_end'].dropna()
    
    print(f"Mean overlap duration: {valid_durations.mean():.1f} years")
    print(f"Median overlap duration: {valid_durations.median():.1f} years")
    print(f"Min overlap duration: {valid_durations.min():.1f} years")
    print(f"Max overlap duration: {valid_durations.max():.1f} years")
    
    if len(valid_starts) > 0 and len(valid_ends) > 0:
        print(f"\nEarliest pair start: {valid_starts.min()}")
        print(f"Latest pair end: {valid_ends.max()}")

# Unpaired gauges
valid_unpaired = unpaired_non_reservoir[unpaired_non_reservoir['valid_start'].notna()]
print(f"\nUnpaired gauges with valid data: {len(valid_unpaired)} out of {len(unpaired_non_reservoir)}")

if len(valid_unpaired) > 0:
    unpaired_non_reservoir['duration_days'] = (
        unpaired_non_reservoir['valid_end'] - unpaired_non_reservoir['valid_start']
    ).apply(lambda x: x.days if pd.notna(x) else None)
    unpaired_non_reservoir['duration_years'] = unpaired_non_reservoir['duration_days'] / 365.25
    
    # Filter out None/NaN values for statistics
    valid_durations = unpaired_non_reservoir['duration_years'].dropna()
    
    print(f"Mean duration: {valid_durations.mean():.1f} years")
    print(f"Median duration: {valid_durations.median():.1f} years")

# Display sample pairs
print("\n" + "="*60)
print("SAMPLE PAIRED GAUGES (first 5 with valid data)")
print("="*60)

sample_pairs = paired_reservoir[paired_reservoir['pair_start'].notna()].head()
for _, res_row in sample_pairs.iterrows():
    pair_id = res_row['pair_id']
    non_res_row = paired_non_reservoir[paired_non_reservoir['pair_id'] == pair_id].iloc[0]
    
    print(f"\nPair {pair_id}:")
    print(f"  Reservoir: {res_row['gauge_id']} ({res_row['reservoir_contributing_area']:.1f}%)")
    print(f"  Non-reservoir: {non_res_row['gauge_id']}")
    print(f"  Shared period: {res_row['pair_start'].date()} to {res_row['pair_end'].date()}")
    duration = (res_row['pair_end'] - res_row['pair_start']).days / 365.25
    print(f"  Duration: {duration:.1f} years")

print("\n" + "="*60)
print("Data processing complete!")
print("="*60)

In [ ]:
chosen_catchments = {
    'paired_reservoir': paired_reservoir[['gauge_id', 'pair_start', 'pair_end']],
    'paired_non_reservoir': paired_non_reservoir[['gauge_id', 'pair_start', 'pair_end']],
    'unpaired_non_reservoir': unpaired_non_reservoir[['gauge_id', 'valid_start', 'valid_end']]

}

# with open('/home/mokr/Reservoir_Modelling/chosen_catchments.pkl', 'wb') as f:
#     pickle.dump(chosen_catchments, f)


In [ ]:
def generate_all_catchment_date_pairs(catchments_df, forecast_days=90, catchment_type=None, pair_id_col=None):
    """
    Generate all valid catchment-date pairs for a given DataFrame.
    
    Parameters:
    -----------
    catchments_df : DataFrame
        DataFrame with catchment information and date columns
    forecast_days : int
        Number of days of data required after the initialization date
    catchment_type : str, optional
        Type label for these catchments (e.g., 'paired_reservoir')
    pair_id_col : str, optional
        Name of the pair_id column if it exists
    
    Returns:
    --------
    all_pairs : list of dict
        List of all valid catchment-date pairs
    """
    all_pairs = []
    
    # Determine if this is paired or unpaired data
    has_pair_dates = 'pair_start' in catchments_df.columns and 'pair_end' in catchments_df.columns
    
    forecast_buffer = pd.Timedelta(days=forecast_days)
    
    for _, row in catchments_df.iterrows():
        # Get the date range for this catchment
        if has_pair_dates:
            if pd.notna(row['pair_start']) and pd.notna(row['pair_end']):
                start_date = pd.Timestamp(row['pair_start'])
                end_date = pd.Timestamp(row['pair_end'])
                pair_id = row.get(pair_id_col, None) if pair_id_col else None
            else:
                continue
        else:
            if pd.notna(row['valid_start']) and pd.notna(row['valid_end']):
                start_date = pd.Timestamp(row['valid_start'])
                end_date = pd.Timestamp(row['valid_end'])
                pair_id = None
            else:
                continue
        
        # Calculate the latest valid initialization date
        latest_init_date = end_date - forecast_buffer
        
        if latest_init_date <= start_date:
            # Not enough data for this forecast length, skip
            print(f"Warning: Catchment {row['gauge_id']} doesn't have {forecast_days} days of data")
            continue
        
        # Add all valid dates for this catchment
        num_valid_days = (latest_init_date - start_date).days + 1
        
        for day_offset in range(num_valid_days):
            init_date = start_date + pd.Timedelta(days=day_offset)
            pair_dict = {
                'gauge_id': f'{int(row['gauge_id'])}',
                'pair_id': pair_id,
                'init_date': init_date
            }
            if catchment_type:
                pair_dict['catchment_type'] = catchment_type
            all_pairs.append(pair_dict)
    
    return all_pairs

In [ ]:
paired_reservoir_pairs[0]

In [ ]:
# Generate pairs for each catchment type
forecast_days = 90

print("Generating pairs for paired_reservoir catchments...")
paired_reservoir_pairs = generate_all_catchment_date_pairs(
    paired_reservoir[['gauge_id', 'pair_id', 'pair_start', 'pair_end']],
    forecast_days=forecast_days,
    catchment_type='paired_reservoir',
    pair_id_col='pair_id'
)

print("Generating pairs for paired_non_reservoir catchments...")
paired_non_reservoir_pairs = generate_all_catchment_date_pairs(
    paired_non_reservoir[['gauge_id', 'pair_id', 'pair_start', 'pair_end']],
    forecast_days=forecast_days,
    catchment_type='paired_non_reservoir',
    pair_id_col='pair_id'
)

print("Generating pairs for unpaired_non_reservoir catchments...")
unpaired_non_reservoir_pairs = generate_all_catchment_date_pairs(
    unpaired_non_reservoir[['gauge_id', 'valid_start', 'valid_end']],
    forecast_days=forecast_days,
    catchment_type='unpaired_non_reservoir'
)

# Create the dictionary
catchment_date_pairs = {
    'paired_reservoir': paired_reservoir_pairs,
    'paired_non_reservoir': paired_non_reservoir_pairs,
    'unpaired_non_reservoir': unpaired_non_reservoir_pairs
}

# Print statistics
print(f"\nPaired reservoir pairs: {len(paired_reservoir_pairs)}")
print(f"Paired non-reservoir pairs: {len(paired_non_reservoir_pairs)}")
print(f"Unpaired non-reservoir pairs: {len(unpaired_non_reservoir_pairs)}")
print(f"Total pairs: {sum(len(v) for v in catchment_date_pairs.values())}")

# Save to pickle file
with open('/home/mokr/Reservoir_Modelling/catchment_date_pairs.pkl', 'wb') as f:
    pickle.dump(catchment_date_pairs, f)

print("\nSaved catchment-date pairs to '/home/mokr/Reservoir_Modelling/catchment_date_pairs.pkl'")

In [ ]:
def sample_batch(all_pairs, batch_size, random_seed=None):
    """Sample a batch of catchment-date pairs uniformly."""
    if random_seed is not None:
        np.random.seed(random_seed)
    
    indices = np.random.choice(len(all_pairs), size=batch_size, replace=False)
    return [all_pairs[i] for i in indices]

all_pairs = (catchment_date_pairs['paired_reservoir'] + 
             catchment_date_pairs['paired_non_reservoir'] + 
             catchment_date_pairs['unpaired_non_reservoir'])

# Example usage
batch = sample_batch(all_pairs, batch_size=32)

batch[0]

# Getting data from ids

In [ ]:
with open('/home/mokr/Reservoir_Modelling/Sample_Pairs/catchment_date_pairs.pkl', 'rb') as f:
    catchment_pairings = pickle.load(f)

with open('/home/mokr/Reservoir_Modelling/Normalised_Temporal_Data.pkl', 'rb') as f:
    temporal_data = pickle.load(f)



In [ ]:
static_variables = pd.read_csv('/perm/mokr/Scaled_CamelsGB_V2/normalised_static_data.csv', 
                                dtype={'gauge_id': str})

static_variables = static_variables.set_index('gauge_id')
static_variables = static_variables.select_dtypes(include=[np.number])


In [ ]:
static_variables.loc['1001']

In [ ]:
def sample_batch(all_pairs, batch_size, random_seed=None):
    """Sample a batch of catchment-date pairs uniformly."""
    if random_seed is not None:
        np.random.seed(random_seed)
    
    indices = np.random.choice(len(all_pairs), size=batch_size, replace=False)
    return [all_pairs[i] for i in indices]


# Example usage
batch = sample_batch(catchment_pairings['paired_reservoir'], batch_size=32)

batch[0]

In [ ]:
static_variables = pd.read_csv('/perm/mokr/Scaled_CamelsGB_V2/normalised_static_data.csv', 
                                dtype={'gauge_id': str})



In [ ]:
static_variables[static_variables['gauge_id'] == batch[0]['gauge_id']]

In [ ]:
goals = ['discharge_spec_x', 'discharge_spec_y', 'discharge_vol_x', 'discharge_vol_y'] # Need to choose what we use as goal, using Specific Discharge from CamelsGBV2 for now
goal = 'discharge_spec_x'

## this is the code for a given sample
gauge_id = batch[0]['gauge_id']
init_date = batch[0]['init_date']
end_date = init_date + pd.Timedelta(days=89) # Gives timeseries of days + 1

static_input = static_variables[static_variables['gauge_id'] == gauge_id]
temporal_values = temporal_data[gauge_id].loc[init_date:end_date]
true_discharge = temporal_values[goal]
temporal_input = temporal_values - goals


In [ ]:
sliced_data['discharge_spec_x']

# Making different subsets of reservoirs

In [ ]:

# 59 reservoirs found in CamelsUS, 129 in CamelsGB, 150 in CamelsBR, 24 in CamelsCL
# 300 catchments would be enough, check which ones I can use

id = 'cl'



Reservoir_basins_list = []
ids = ['', 'gb', 'cl', 'br']
for id in ids:
    base_dir = f"/perm/mokr/Caravans/Caravan/attributes/camels{id}/"
    
    # Load the CSV files
    df_hydroatlas = pd.read_csv(base_dir + f"attributes_hydroatlas_camels{id}.csv")
    filtered_catchments = df_hydroatlas[df_hydroatlas['dor_pc_pva'] != 0]
    Reservoir_basins_list.append(filtered_catchments['gauge_id'].to_list())
    
    # Drop NaNs for accurate counting
    values = df_hydroatlas['dor_pc_pva'].dropna()
    values = values[values != 0]
    print(len(values))

Reservoir_basins_list = [item for sublist in Reservoir_basins_list for item in sublist]
# Count frequency of each unique value (rounded if needed)
frequency = values.value_counts().sort_index()
    
# Plot the frequency as a bar chart
# plt.figure(figsize=(10, 6))
# frequency.plot(kind='bar')
# plt.title("Frequency of 'dor_pc_pva' Values")
# plt.xlabel("dor_pc_pva")
# plt.ylabel("Frequency")
# plt.grid(True)
# plt.tight_layout()

# plt.show()

# with open('/home/mokr/Reservoir_Modelling/Dor_Reservoirs_My_Susbet.txt', 'w') as f:
#     for item in Reservoir_basins_list:
#         f.write(f"{item}\n")

In [ ]:
import os
import pandas as pd

directory = '/ec/res4/scratch/ecm7072/caravan/attributes'
reservoir_basins_list = []

# Loop over all subdirectories
for region in os.listdir(directory):
    base_dir = os.path.join(directory, region)
    if not os.path.isdir(base_dir):
        continue  # Skip files, only process directories

    csv_path = os.path.join(base_dir, f"attributes_hydroatlas_{region}.csv")
    


    # Load and filter the CSV
    df_hydroatlas = pd.read_csv(csv_path)
    filtered_catchments = df_hydroatlas[df_hydroatlas['dor_pc_pva'] != 0]
    reservoir_basins_list.append(filtered_catchments['gauge_id'].tolist())

    # Optional: Print some info
    values = df_hydroatlas['dor_pc_pva'].dropna()
    values = values[values != 0]
    print(f"Region: {region}, Count with DOR > 0: {len(values)}")


marias_reservoir_basins_list = [item for sublist in reservoir_basins_list for item in sublist]
with open('/home/mokr/Reservoir_Modelling/Dor_Reservoirs_Marias_Susbet.txt', 'w') as f:
    for item in marias_reservoir_basins_list:
        f.write(f"{item}\n")

# Comparing/exploring reservoir subsets

In [ ]:
# Get the lists of each catchment
# Step 1: Read gauge IDs from text files
with open('/home/mokr/Reservoir_Modelling/Marias_Reservoir_Subset.txt', 'r') as f:
    Marias_Reservoirs = [line.strip() for line in f if line.strip()]

with open('/home/mokr/Reservoir_Modelling/Dor_Reservoirs_Marias_Susbet.txt', 'r') as f:
    All_DoR_Reservoirs = [line.strip() for line in f if line.strip()]

with open('/home/mokr/Reservoir_Modelling/Dor_Reservoirs_My_Susbet.txt', 'r') as f:
    My_DoR_Reservoirs = [line.strip() for line in f if line.strip()]

with open('/home/mokr/Reservoir_Modelling/Google_Subset.txt', 'r') as f:
    Google_Reservoirs = [line.strip() for line in f if line.strip()]



Marias_Reservoirs = [
    item.lower().removeprefix("caravan_")
    for item in Marias_Reservoirs
    if not item.strip().startswith("#")
]

# Step 2: Combine all lists to identify needed CAMELS regions
Reservoirs_Lists = [Marias_Reservoirs, All_DoR_Reservoirs, My_DoR_Reservoirs, Google_Reservoirs]
Camels_Regions = set()

In [ ]:
# Paths to reservoir ID lists
for reservoir_list in Reservoirs_Lists:
    for rid in reservoir_list:
        if '_' in rid:
            region = rid.split('_')[0]
            Camels_Regions.add(region)

Camels_Regions = {region.lower() for region in Camels_Regions}
Camels_Regions.remove('hybas')

In [ ]:
# Step 2: Load attribute files per region
Camels_Attributes = {}
Camels_Hydroatlas = {}
for region in Camels_Regions:
    attributes_path = f'/hpcperm/mokr/Marias_caravan/attributes/{region}/attributes_other_{region}.csv'
    hydroatlas_path = f'/hpcperm/mokr/Marias_caravan/attributes/{region}/attributes_hydroatlas_{region}.csv'

    
    Camels_Attributes[region] = pd.read_csv(attributes_path)
    Camels_Hydroatlas[region] = pd.read_csv(hydroatlas_path)

# Step 4: Build attribute DataFrames for each list
Reservoir_DataFrames_Attr = []
Reservoir_DataFrames_HydroAtlas = []

for reservoir_list in Reservoirs_Lists:
    df_parts_attr = []
    df_parts_hydro = []

    for rid in reservoir_list:
        if '_' not in rid:
            continue
        region = rid.split('_')[0]
        attr_df = Camels_Attributes.get(region)
        hydroatlas_df = Camels_Hydroatlas.get(region)
        if attr_df is not None:
            match_attr = attr_df[attr_df['gauge_id'] == rid]
            match_hydro = hydroatlas_df[hydroatlas_df['gauge_id'] == rid]
            if not match_attr.empty:
                df_parts_attr.append(match_attr)
                df_parts_hydro.append(match_hydro)

    df_combined = pd.concat(df_parts_attr, ignore_index=True) if df_parts_attr else pd.DataFrame()
    df_combined_hydroatlas = pd.concat(df_parts_hydro, ignore_index=True) if df_parts_hydro else pd.DataFrame()

    Reservoir_DataFrames_Attr.append(df_combined)
    Reservoir_DataFrames_HydroAtlas.append(df_combined_hydroatlas)

# Step 5: Unpack results
df_marias, df_all_dor, df_my_dor, df_google = Reservoir_DataFrames_Attr
hydro_df_marias, hydro_df_all_dor, hydro_df_my_dor, hydro_df_google = Reservoir_DataFrames_HydroAtlas

In [ ]:
df_combined

In [ ]:

from scipy import stats

# Assuming hydro_df_marias_dor is your DataFrame
# Extract the column data
data = hydro_df_all_dor['dor_pc_pva'].dropna()  # Remove any NaN values

# Method 1: Manual CDF calculation
def create_empirical_cdf(data):
    """Create empirical CDF from data"""
    sorted_data = np.sort(data)
    n = len(sorted_data)
    y_values = np.arange(1, n + 1) / n
    return sorted_data, y_values

# Create the CDFs
x_emp, y_emp = create_empirical_cdf(data)

# Plot empirical CDF

plt.plot(x_emp, y_emp, 'b-', linewidth=2, label='Empirical CDF')
plt.xlabel('dor_pc_pva values')
plt.ylabel('Cumulative Probability')
plt.title('Empirical CDF')
plt.grid(True, alpha=0.3)
plt.legend()
plt.xscale('log')
plt.tight_layout()
plt.show()

In [ ]:
import math

dataframes = {
    'Marias Subset': hydro_df_marias,
    'Catchments with DoR>0': hydro_df_all_dor,
    'Googles Subset': hydro_df_google,
}



def create_cdf_grid(dataframes_dict, column_name='dor_pc_pva'):
    """Create a grid of CDF plots for multiple dataframes"""
    
    n_dfs = len(dataframes_dict)
    
    # Calculate grid dimensions
    n_cols = min(3, n_dfs)  # Maximum 3 columns
    n_rows = math.ceil(n_dfs / n_cols)
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    
    # Handle case where there's only one subplot
    if n_dfs == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Flatten axes array for easier indexing
    axes_flat = axes.flatten() if n_dfs > 1 else axes
    
    # Store all data for global statistics
    all_data = {}
    
    # Create CDF for each dataframe
    for idx, (df_name, df) in enumerate(dataframes_dict.items()):
        # Extract and clean data
        if column_name in df.columns:
            data = df[column_name].dropna()
        else:
            print(f"Warning: Column '{column_name}' not found in {df_name}")
            continue
            
        if len(data) == 0:
            print(f"Warning: No valid data in {df_name}")
            continue
            
        # Store data for later use
        all_data[df_name] = data
        
        # Create empirical CDF
        x_values, cdf_values = create_empirical_cdf(data)
        
        # Plot on appropriate subplot
        ax = axes_flat[idx]
        ax.plot(x_values, cdf_values, 'b-', linewidth=2, label='Empirical CDF')
        ax.set_xlabel('Degree of Regulation (%)')
        ax.set_ylabel('Cumulative Probability')
        ax.set_title(f'Spread of Degree of Regulation for {df_name}')
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
        ax.legend()
        
        # Add some statistics as text
        stats_text = f'n={len(data)}\nμ={data.mean():.2f}\nσ={data.std():.2f}'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Hide any unused subplots
    for idx in range(len(dataframes_dict), len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("Summary Statistics:")
    print("-" * 60)
    for df_name, data in all_data.items():
        print(f"\n{df_name}:")
        print(f"  Count: {len(data)}")
        print(f"  Mean: {data.mean():.4f}")
        print(f"  Std: {data.std():.4f}")
        print(f"  Min: {data.min():.4f}")
        print(f"  Max: {data.max():.4f}")
        print(f"  Median: {data.median():.4f}")
        print(f"  25th percentile: {data.quantile(0.25):.4f}")
        print(f"  75th percentile: {data.quantile(0.75):.4f}")

# Create the grid of CDF plots
create_cdf_grid(dataframes, 'dor_pc_pva')

# Make map of basins I'm using

In [ ]:
def load_list(filepath):
    with open(filepath, "r") as f:
        return [line.strip() for line in f.readlines()]

# Directory where files are saved
path = "/home/mokr/Reservoir_Modelling/Catchment_Lists"

# Load each group
Reservoirs_Train = load_list(os.path.join(path, "Reservoirs_Train.txt"))
Reservoirs_Val = load_list(os.path.join(path, "Reservoirs_Val.txt"))
Reservoirs_Test = load_list(os.path.join(path, "Reservoirs_Test.txt"))

NonReservoirs_Train = load_list(os.path.join(path, "NonReservoirs_Train.txt"))
NonReservoirs_Val = load_list(os.path.join(path, "NonReservoirs_Val.txt"))
NonReservoirs_Test = load_list(os.path.join(path, "NonReservoirs_Test.txt"))

Basin_Lists = [Reservoirs_Train, Reservoirs_Val, Reservoirs_Test, NonReservoirs_Train, NonReservoirs_Val, NonReservoirs_Test]

In [ ]:
import pandas as pd
flat_basin_list = [gauge_id for sublist in Basin_Lists for gauge_id in sublist]


# --- 3. Filter the DataFrame using .isin() ---
df_combined = df_combined[df_combined['gauge_id'].isin(flat_basin_list)]



In [ ]:
df_combined

In [ ]:
# --- 1. Setup Data ---

# The path to your shapefile
shapefile_path = "/home/mokr/GLOFAS_ML_Flood_Modelling/data/ne_110m_admin_0_countries/raw/ne_110m_admin_0_countries.shp"

# Load the world map
world = gpd.read_file(shapefile_path)
Koppen_Geiger_Path = '/home/mokr/Loss_Functions_Paper/Kopen_Geiger/c1976_2000.shp'

KG = gpd.read_file(Koppen_Geiger_Path)
KG_dissolved = KG.dissolve(by='GRIDCODE')
KG_dissolved = KG_dissolved.to_crs("EPSG:4326")
KG_dissolved['classification_area'] = KG_dissolved.area

KG = KG.to_crs("EPSG:4326")
KG['area'] = KG.area.sum()

# --- 2. Convert Gauge DataFrame to GeoDataFrame ---

# Note: Correcting 'gauge_lot' to 'gauge_lon'.
# Create a GeoDataFrame from your lat/lon columns.
gdf_gauges = gpd.GeoDataFrame(
    df_combined,
    geometry=gpd.points_from_xy(df_combined['gauge_lon'], df_combined['gauge_lat']),
    crs="EPSG:4326"  # Set the coordinate reference system to WGS84
)



In [ ]:
from matplotlib.patches import Patch
from shapely.geometry import Polygon


# --- 2. Define a Function to Map GridCode to Color ---
def get_color_from_gridcode(grid_code):
    """
    Generates an RGBA color based on a GridCode using standard
    Köppen-Geiger colors.
    """
    # Define the standard Köppen-Geiger base colors
    base_colors = {
        1: '#87b7e7',  # Medium-Light Blue (A: Tropical)
        2: '#ff8a8a',  # Soft Red/Coral    (B: Arid)
        3: '#82f29b',  # Soft Green        (C: Temperate)
        4: '#b3a1e1',  # Medium Lavender   (D: Continental)
        5: '#60e7e5',  # Soft Turquoise    (E: Polar)
        6: '#bebebe'   # Medium Grey       (H: Highland/Other)
    }

    major_type = grid_code // 10
    subtype = grid_code % 10
    color = base_colors.get(major_type, 'black') # Default to black if not found

    # Map the subtype (0-9) to an alpha value (shade)
    min_alpha = 0.2
    max_alpha = 1.0
    alpha = 1 - (min_alpha + (subtype / 9) * (max_alpha - min_alpha) )

    return (color, alpha)

# --- 3. Apply the Function and Plot ---

KG[['color', 'alpha']] = KG['GRIDCODE'].apply(
    get_color_from_gridcode
).apply(pd.Series)


In [ ]:
# --- 3. Plotting ---

# Create a figure and axes for the plot
fig, ax = plt.subplots(1, 1, figsize=(15, 10))

# Plot the world map as the base layer
world.plot(ax=ax, color='lightgray', edgecolor='black')

KG.plot(ax=ax, color=KG['color'], alpha=KG['alpha'], edgecolor='gray', linewidth=0.5)

# --- 4. Create a Custom Legend with Climate Names ---

# Update legend labels to be more descriptive
legend_labels = {
    1: 'A: Tropical',
    2: 'B: Arid',
    3: 'C: Temperate',
    4: 'D: Continental',
    6: 'E: Polar'
}

legend_elements = [Patch(facecolor=get_color_from_gridcode(k*10)[0], label=v)
    for k, v in legend_labels.items()]


# Plot the gauge locations on top of the map
gdf_gauges.plot(ax=ax, color='black', marker='o', markersize=1, label='Gauges')

# Customize the plot
ax.set_title('World Map with Gauge Locations')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import os

# camelses_9092 is one removed with no regulation, plot it to figure out if streamflow is weird
for catchment in hydro_df_marias['gauge_id']:
    try:
        region = catchment.split('_')[0]
        river = catchment.split('_')[1]
        
        # Construct file path
        file_path = f'/hpcperm/mokr/Marias_caravan/timeseries/netcdf/{region}/{region}_{river}.nc'
        
        # Check if file exists
        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue
            
        # Load the NetCDF file
        timeseries = xr.open_dataset(file_path)
        
        # Convert to DataFrame and reset index to handle multi-index issues
        timeseries_df = timeseries.to_dataframe().reset_index()
        
        # Check if streamflow column exists
        if 'streamflow' not in timeseries_df.columns:
            print(f"No 'streamflow' column found in {catchment}. Available columns: {list(timeseries_df.columns)}")
            timeseries.close()
            continue
            
        # Remove any NaN values and get the last 365 non-null values
        streamflow_data = timeseries_df['streamflow'].dropna()
        
        if len(streamflow_data) < 365:
            print(f"Not enough data for {catchment}: only {len(streamflow_data)} points available")
            streamflow_last_365 = streamflow_data
        else:
            streamflow_last_365 = streamflow_data.iloc[-365:]
        
        # Create the plot
        plt.figure(figsize=(12, 6))
        plt.plot(range(len(streamflow_last_365)), streamflow_last_365.values, linewidth=1.5, color='steelblue')
        plt.title(f'Final {len(streamflow_last_365)} Days of Streamflow Data - {catchment}', fontsize=14, fontweight='bold')
        plt.xlabel('Days from End', fontsize=12)
        plt.ylabel('Streamflow', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        plt.show()
        
        # Close the dataset to free memory
        timeseries.close()
        
    except Exception as e:
        print(f"Error processing {catchment}: {str(e)}")
        continue

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Load the world map
world = gpd.read_file("/home/mokr/GLOFAS_ML_Flood_Modelling/data/ne_110m_admin_0_countries/raw/ne_110m_admin_0_countries.shp")

# Create GeoDataFrames for each reservoir dataset
def create_gdf(df, color):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['gauge_lon'], df['gauge_lat']),
        crs="EPSG:4326"
    )
    gdf['color'] = color
    return gdf

gdf_marias = create_gdf(df_marias, 'red')
# gdf_marias_dor = create_gdf(df_marias_dor, 'blue')
gdf_my_dor = create_gdf(df_my_dor, 'green')
gdf_google = create_gdf(df_google, 'orange')

# Plotting
fig, ax = plt.subplots(figsize=(14, 8))
world.plot(ax=ax, color='lightgrey', edgecolor='white')

alpha = 0.7
gdf_marias.plot(ax=ax, markersize=1, color='red', alpha = alpha, label='Marias')
gdf_marias_dor.plot(ax=ax, markersize=1, color='blue', alpha = alpha, label='Marias DoR')
gdf_my_dor.plot(ax=ax, markersize=1, color='green', alpha = alpha, label='My DoR')
# gdf_google.plot(ax=ax, markersize=1, color='orange', alpha = alpha, label='Google Subset')


plt.legend()
plt.title("Reservoir Gauge Locations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Define bins and labels
bins = [0, 50, 500, 5000, 100000, 1e6, float('inf')]
labels = ['<50', '50 - 500', '500–5,000', '5,000–100,000', '>100,000', '1e6']

def bin_and_count(df):
    # Clean and bin the area column
    df = df.dropna(subset=['area'])
    df = df[df['area'] > 0]
    binned = pd.cut(df['area'], bins=bins, labels=labels, right=True)
    counts = binned.value_counts().sort_index()
    proportions = counts / counts.sum()
    return counts, proportions

# Get counts and proportions for each group
counts_marias, props_marias = bin_and_count(df_marias)
counts_marias_dor, props_marias_dor = bin_and_count(df_marias_dor)
counts_my_dor, props_my_dor = bin_and_count(df_my_dor)
counts_google, props_google = bin_and_count(df_google)


# Combine proportions into one DataFrame
proportions_df = pd.DataFrame({
    f'Marias shortlist {sum(counts_marias)}': props_marias,
    f'Large Set Selected Catchments {sum(counts_marias_dor)}': props_marias_dor,
    f'My Selected Catchments {sum(counts_my_dor)}': props_my_dor,
    f'Google Selected Catchments {sum(counts_google)}': props_google

})

# Also combine absolute counts for labeling
counts_df = pd.DataFrame({
    f'Marias {sum(counts_marias)}': counts_marias,
    f'Marias DoR {sum(counts_marias_dor)}': counts_marias_dor,
    f'My DoR {sum(counts_my_dor)}': counts_my_dor,
    f'Google {sum(counts_google)}': counts_google

})

# Plot
ax = proportions_df.plot(kind='bar', figsize=(10, 6), rot=0)
ax.set_title('Proportion of Reservoirs by Catchment Area Range')
ax.set_xlabel('Catchment Area Range (km²)')
ax.set_ylabel('Proportion of Reservoirs')
plt.legend(title="Reservoir List")


plt.tight_layout()
plt.show()



In [ ]:
# Extract gauge ID sets
# df_marias, df_marias_dor, df_my_dor, df_google
set1 = set(df_marias['gauge_id'])
set2 = set(df_marias_dor['gauge_id'])
set3 = set(df_google['gauge_id'])

# Plot Venn diagram
plt.figure(figsize=(8, 6))
venn3([set1, set2, set3], set_labels=('Marias', 'My DoR', 'Google'))
plt.title("Venn Diagram of Reservoir Catchments")
plt.tight_layout()
plt.show()

# Make Train/Val/Test and Reservoir dominated/not dominated from Google dataset:

- Have a threshold of dor-pc-pva > 100 for reservoir
- Set = 0 for non reservoir
- Also need to preserve using the whole dataset for the 'default' LSTM

In [ ]:
Reservoirs = hydro_df_google[hydro_df_google['dor_pc_pva'] > 800]['gauge_id'].to_list()

Reservoirs = hydro_df_google[(hydro_df_google['dor_pc_pva'] >= 2) & 
                            (hydro_df_google['dor_pc_pva'] <= 800)
                            ]['gauge_id'].to_list()

Non_Reservoirs = hydro_df_google[hydro_df_google['dor_pc_pva'] == 0]['gauge_id'].to_list()

In [ ]:
len(Reservoirs)

In [ ]:
import random 

# Helper to split a list into train/val/test
def split_list(data, train_frac=0.7, val_frac=0.15):
    random.shuffle(data)
    n = len(data)
    train_end = int(n * train_frac)
    val_end = train_end + int(n * val_frac)
    return data[:train_end], data[train_end:val_end], data[val_end:]

# Split each group
res_train, res_val, res_test = split_list(Reservoirs)
nonres_train, nonres_val, nonres_test = split_list(Non_Reservoirs)

# Output directory
outdir = "/home/mokr/Reservoir_Modelling/Catchment_Lists"
os.makedirs(outdir, exist_ok=True)

# Function to save a list to .txt
def save_list(data, filename):
    with open(filename, "w") as f:
        for item in data:
            f.write(f"{item}\n")

In [ ]:
save_list(res_train,   os.path.join(outdir, "Reservoirs_Train_DOR_2_to_800.txt"))
save_list(res_val,     os.path.join(outdir, "Reservoirs_Val_DOR_2_to_800.txt"))
save_list(res_test,    os.path.join(outdir, "Reservoirs_Test_DOR_2_to_800.txt"))

In [ ]:
# Save each group
save_list(res_train,   os.path.join(outdir, "Reservoirs_Train.txt"))
save_list(res_val,     os.path.join(outdir, "Reservoirs_Val.txt"))
save_list(res_test,    os.path.join(outdir, "Reservoirs_Test.txt"))
save_list(nonres_train,os.path.join(outdir, "NonReservoirs_Train.txt"))
save_list(nonres_val,  os.path.join(outdir, "NonReservoirs_Val.txt"))
save_list(nonres_test, os.path.join(outdir, "NonReservoirs_Test.txt"))